<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">


<br><br><br><br>

<h1><font color="#00586D" size=5>Despliegue del modelo entrenado de Yolov8 en un endpoint de Sagemaker</font></h1>
<br><br>


<div align="right">
<font color="#00586D" size=3>Javier de la Ossa</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube</font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>

</div>

---
<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Compresión del modelo y la carpeta `code/` para almacenamiento en S3](#section1)
* [2. Creación del modelo PyTorch para su implementación en Sagemaker](#section2)
* [3. Despliegue del modelo creado de PyTorch en un endpoint de Sagemaker](#section3)

<br>

---

In [1]:
import os, sagemaker, subprocess, boto3
from datetime import datetime
from sagemaker import s3
from sagemaker import get_execution_role
from sagemaker.pytorch import PyTorchModel
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serverless import ServerlessInferenceConfig

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


---

<a id="section1"></a>
## <font color="#00586D"> 1. Compresión del modelo y la carpeta `code/` para almacenamiento en S3</font>
<br>

In [2]:
model_name = "last.pt"

bashCommand = f"tar -czvf model.tar.gz code/ {model_name}"
process = subprocess.Popen(bashCommand.split(), stdout=subprocess.PIPE)
output, error = process.communicate()

In [3]:
s3_client = boto3.client("s3")
response = s3_client.list_buckets()
for bucket in response["Buckets"]:
    if "yolov8" in bucket["Name"]:
        bucket = "s3://" + bucket["Name"]
        break

print(f"Bucket: {bucket}")
sess = sagemaker.Session(default_bucket=bucket.split("s3://")[-1])

prefix = "yolov8/receipts-model-endpoint"

Bucket: s3://sm-receipts-yolov8-jadelaossa


In [4]:
sm_client = boto3.client(service_name="sagemaker")
runtime_sm_client = boto3.client(service_name="sagemaker-runtime")

account_id = boto3.client("sts").get_caller_identity()["Account"]
region = boto3.Session().region_name

role = get_execution_role()
print(f"Role: {role}")

model_data = s3.S3Uploader.upload("model.tar.gz", bucket + "/" + prefix)
print(f"Model Data: {model_data}")

Role: arn:aws:iam::637423468312:role/LabRole
Model Data: s3://sm-receipts-yolov8-jadelaossa/yolov8/receipts-model-endpoint/model.tar.gz


<a id="section2"></a>
## <font color="#00586D"> 2. Creación del modelo PyTorch para su implementación en Sagemaker</font>
<br>

In [5]:
model = PyTorchModel(entry_point="inference.py",
                     model_data=model_data, 
                     framework_version="1.12", 
                     py_version="py38",
                     role=role,
                     env={"TS_MAX_RESPONSE_SIZE":"20000000", "YOLOV8_MODEL": model_name},
                     sagemaker_session=sess)

In [6]:
# Configure serverless inference
serverless_config = ServerlessInferenceConfig(
  memory_size_in_mb=4096,
  max_concurrency=1
)

<a id="section3"></a>
## <font color="#00586D"> 3. Despliegue del modelo creado de PyTorch en un endpoint de Sagemaker</font>
<br>

In [7]:
INSTANCE_TYPE = "ml.m5.xlarge"
ENDPOINT_NAME = "yolov8-pytorch-" + str(datetime.utcnow().strftime("%Y-%m-%d-%H-%M-%S-%f"))

# Store the endpoint name in the history to be accessed by 2-test-sm-endpoint.ipynb notebook
%store ENDPOINT_NAME
print(f"Endpoint Name: {ENDPOINT_NAME}")

predictor = model.deploy(initial_instance_count=1, 
                         instance_type=INSTANCE_TYPE,
                         deserializer=JSONDeserializer(),
                         serverless_inference_config=serverless_config,
                         endpoint_name=ENDPOINT_NAME)

Stored 'ENDPOINT_NAME' (str)
Endpoint Name: yolov8-pytorch-2024-08-24-12-07-51-511813
----!